# MiroFish fully inside Google Colab

This notebook has **no Zep account, Zep API key, Gemini key, or external inference service**. It runs a local-first MiroFish fork with Graphiti + Neo4j in the Colab VM, `LiquidAI/LFM2.5-350M` on the Colab GPU for chat/completions, and `BAAI/bge-small-en-v1.5` locally for embeddings.

It does download public code/model weights from GitHub and Hugging Face at setup time, and Cloudflare Quick Tunnels expose the UI to your browser. Apart from those downloads and the temporary browser tunnel, seed data, graph data, embeddings, and LLM prompts remain in the Colab runtime.

## Important design change

The upstream `666ghj/MiroFish` requires Zep Cloud. This notebook therefore uses `tt-a1i/MiroFish-local`, a local-first fork that provides `ZEP_BACKEND=graphiti` and replaces Zep with locally running Graphiti + Neo4j. It is not the upstream Docker image.

Colab is still temporary: a runtime reset destroys all local data and URLs. Do not use this as production hosting or upload secrets/confidential documents to an unprotected tunnel.

## 1. Enable a GPU

In Colab choose **Runtime → Change runtime type → T4 GPU**, reconnect, then run this cell. A GPU is required for usable LLM latency. No credentials are requested anywhere in this notebook.

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('No NVIDIA GPU. Enable a Colab T4 GPU, reconnect, then rerun.')
print('Detected:', r.stdout.strip())

## 2. Local LLM: LFM2.5-350M via vLLM

vLLM exposes an OpenAI-compatible API only on `127.0.0.1:8000`; it is never tunneled publicly. The initial 8,192-token context is deliberately conservative for Colab Free. Increase it only after the full pipeline works.

In [ ]:
%%bash
set -euo pipefail
pip -q install -U vllm
mkdir -p /content/local-mirofish/logs
pkill -f 'vllm.entrypoints.openai.api_server' 2>/dev/null || true
nohup python -m vllm.entrypoints.openai.api_server \
  --model LiquidAI/LFM2.5-350M \
  --served-model-name LiquidAI/LFM2.5-350M \
  --dtype bfloat16 --max-model-len 8192 --gpu-memory-utilization 0.65 \
  --host 127.0.0.1 --port 8000 \
  > /content/local-mirofish/logs/vllm.log 2>&1 &
for i in $(seq 1 180); do
  curl -fsS --max-time 3 http://127.0.0.1:8000/v1/models && exit 0
  sleep 2
done
tail -n 120 /content/local-mirofish/logs/vllm.log
exit 1

## 3. Local embedding API

Graphiti needs an OpenAI-compatible `/v1/embeddings` endpoint as well as a chat endpoint. This tiny local FastAPI service loads the compact BGE embedding model on the same GPU. A router in the next cell presents both services at one OpenAI-style base URL because MiroFish-Local maps its Graphiti and application LLM settings to the same base URL.

In [ ]:
%%bash
set -euo pipefail
pip -q install -U sentence-transformers fastapi uvicorn httpx
cat > /content/local-mirofish/embed_server.py <<'PY'
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
import time

model = SentenceTransformer('BAAI/bge-small-en-v1.5', device='cuda')
app = FastAPI()
class EmbeddingRequest(BaseModel):
    input: str | list[str]
    model: str | None = None
    encoding_format: str | None = None
@app.get('/health')
def health(): return {'ok': True}
@app.post('/v1/embeddings')
def embeddings(req: EmbeddingRequest):
    texts = [req.input] if isinstance(req.input, str) else req.input
    vectors = model.encode(texts, normalize_embeddings=True).tolist()
    return {'object':'list','data':[{'object':'embedding','embedding':v,'index':i} for i,v in enumerate(vectors)],'model':'BAAI/bge-small-en-v1.5','usage':{'prompt_tokens':0,'total_tokens':0}}
PY
nohup uvicorn --app-dir /content/local-mirofish embed_server:app --host 127.0.0.1 --port 8001 > /content/local-mirofish/logs/embed.log 2>&1 &
for i in $(seq 1 90); do
  curl -fsS --max-time 3 http://127.0.0.1:8001/health && break
  sleep 2
done
curl -fsS http://127.0.0.1:8001/health


In [ ]:
%%bash
set -euo pipefail
cat > /content/local-mirofish/openai_router.py <<'PY'
from fastapi import FastAPI, Request
from fastapi.responses import Response
import httpx
app = FastAPI()
@app.api_route('/{path:path}', methods=['GET','POST','PUT','PATCH','DELETE'])
async def route(path: str, request: Request):
    target = 'http://127.0.0.1:8001' if path == 'v1/embeddings' else 'http://127.0.0.1:8000'
    async with httpx.AsyncClient(timeout=600) as client:
        upstream = await client.request(request.method, f'{target}/{path}', params=request.query_params, content=await request.body(), headers={k:v for k,v in request.headers.items() if k.lower() != 'host'})
    return Response(content=upstream.content, status_code=upstream.status_code, headers={'content-type':upstream.headers.get('content-type','application/json')})
PY
nohup uvicorn --app-dir /content/local-mirofish openai_router:app --host 127.0.0.1 --port 9000 > /content/local-mirofish/logs/router.log 2>&1 &
sleep 3
curl -fsS http://127.0.0.1:9000/v1/models
echo
curl -fsS http://127.0.0.1:9000/v1/embeddings -H 'content-type: application/json' -d '{"input":"local graph embedding test","model":"BAAI/bge-small-en-v1.5"}' | head -c 200
echo

## 4. Verify a completely local model response

This confirms the routing layer can serve the exact OpenAI-chat protocol that MiroFish uses. The request is local to this Colab runtime.

In [ ]:
import requests
payload = {'model':'LiquidAI/LFM2.5-350M','messages':[{'role':'system','content':'Reply only with JSON.'},{'role':'user','content':'Return {"local":true}'}], 'temperature':0.1, 'max_tokens':32, 'response_format':{'type':'json_object'}}
response = requests.post('http://127.0.0.1:9000/v1/chat/completions', json=payload, timeout=120)
response.raise_for_status()
print(response.json()['choices'][0]['message']['content'])

## 5. Docker and local Neo4j

Neo4j holds the Graphiti knowledge graph in the Colab VM. It binds to loopback only, so neither Bolt nor the Neo4j web UI is publicly exposed. The password below is VM-local and temporary; it is not a production credential.

In [ ]:
%%bash
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get -qq update
apt-get -qq install -y docker.io git curl ca-certificates
if ! docker info >/dev/null 2>&1; then
  nohup dockerd > /content/local-mirofish/logs/dockerd.log 2>&1 &
  timeout 75 bash -c 'until docker info >/dev/null 2>&1; do sleep 2; done'
fi
docker rm -f mirofish-neo4j 2>/dev/null || true
docker volume rm mirofish_neo4j_data 2>/dev/null || true
docker run -d --name mirofish-neo4j --restart=no \
  -p 127.0.0.1:7474:7474 -p 127.0.0.1:7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  -e NEO4J_PLUGINS='["apoc"]' \
  -v mirofish_neo4j_data:/data \
  neo4j:5.26
for i in $(seq 1 60); do
  curl -fsS --max-time 3 http://127.0.0.1:7474 >/dev/null && echo 'Neo4j ready' && exit 0
  sleep 2
done
docker logs --tail 100 mirofish-neo4j
exit 1

## 6. Get the local-first MiroFish fork and install it

This fork is required: the upstream MiroFish backend hard-depends on Zep Cloud. The install can take several minutes. The script asks `uv` for the **Graphiti extra**, then checks that the Graphiti package was actually installed rather than silently running an incomplete backend.

The Graphiti and OASIS dependency extras can conflict in some revisions of the fork. If the extra install fails, use the repository's currently documented Graphiti branch/version rather than falling back to Zep.

In [ ]:
%%bash
set -euo pipefail
curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
apt-get -qq install -y nodejs
pip -q install -U uv
rm -rf /content/MiroFish-local
git clone --depth 1 https://github.com/tt-a1i/MiroFish-local.git /content/MiroFish-local
cd /content/MiroFish-local
npm run setup
cd backend
uv sync --extra graphiti
uv run python -c 'import graphiti_core; print("Graphiti installed")'

## 7. Configure Graphiti mode — no Zep variables

`ZEP_BACKEND=graphiti` is the decisive setting. `OPENAI_*` points to the all-local router. `GRAPHITI_EMBEDDING_MODEL` goes to the local BGE service, and `GRAPHITI_LLM_MODEL` goes to local LFM through vLLM. No `ZEP_API_KEY` is written.

The backend and frontend run on the Colab host, so they access local services at `127.0.0.1`, not `host.docker.internal`.

In [ ]:
%%bash
set -euo pipefail
cat > /content/MiroFish-local/.env <<'EOF'
LLM_API_KEY=local
LLM_BASE_URL=http://127.0.0.1:9000/v1
LLM_MODEL_NAME=LiquidAI/LFM2.5-350M
ZEP_BACKEND=graphiti
NEO4J_URI=bolt://127.0.0.1:7687
NEO4J_USER=neo4j
NEO4J_PASSWORD=password
OPENAI_API_KEY=local
OPENAI_BASE_URL=http://127.0.0.1:9000/v1
GRAPHITI_LLM_MODEL=LiquidAI/LFM2.5-350M
GRAPHITI_EMBEDDING_MODEL=BAAI/bge-small-en-v1.5
FLASK_DEBUG=false
EOF
grep -E '^(LLM_|ZEP_BACKEND|NEO4J_|OPENAI_|GRAPHITI_|FLASK_DEBUG)' /content/MiroFish-local/.env

## 8. Start the MiroFish backend and create its tunnel

The backend starts first. Its temporary public address is passed to Vite when starting the frontend; this prevents the browser from trying to call its own `localhost:5001`.

In [ ]:
%%bash
set -euo pipefail
cd /content/MiroFish-local
pkill -f 'flask run' 2>/dev/null || true
nohup npm run backend > /content/local-mirofish/logs/mirofish-backend.log 2>&1 &
for i in $(seq 1 90); do
  curl -sS --max-time 3 http://127.0.0.1:5001/ >/dev/null && break
  sleep 2
done
curl -fsSL -o /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
dpkg -i /tmp/cloudflared.deb
nohup cloudflared tunnel --url http://127.0.0.1:5001 > /content/local-mirofish/logs/tunnel-backend.log 2>&1 &
for i in $(seq 1 30); do
  BACKEND_URL=$(grep -Eo 'https://[-a-z0-9]+\.trycloudflare\.com' /content/local-mirofish/logs/tunnel-backend.log | head -1 || true)
  [ -n "${BACKEND_URL}" ] && break
  sleep 2
done
test -n "${BACKEND_URL:-}"
echo "$BACKEND_URL" > /content/local-mirofish/backend_url.txt
echo "Backend tunnel: $BACKEND_URL"
tail -n 30 /content/local-mirofish/logs/mirofish-backend.log

## 9. Start the frontend and open MiroFish

Open the printed frontend address. It is an unprotected, temporary development link. Keep the Colab session active. For the first complete test, use a short TXT/Markdown seed, 3 agents, and 3 rounds.

A 350M model can be useful for validating the fully local pipeline but is likely to be weak at rich ontology extraction and long multi-agent reports. If Graphiti fails on malformed structured output, reduce the source length and model temperature/agent settings before assuming Neo4j is broken.

In [ ]:
%%bash
set -euo pipefail
cd /content/MiroFish-local
BACKEND_URL=$(cat /content/local-mirofish/backend_url.txt)
pkill -f 'vite' 2>/dev/null || true
VITE_API_BASE_URL="$BACKEND_URL" nohup npm run frontend -- --host 127.0.0.1 > /content/local-mirofish/logs/mirofish-frontend.log 2>&1 &
for i in $(seq 1 90); do
  curl -fsS --max-time 3 http://127.0.0.1:3000/ >/dev/null && break
  sleep 2
done
nohup cloudflared tunnel --url http://127.0.0.1:3000 > /content/local-mirofish/logs/tunnel-frontend.log 2>&1 &
for i in $(seq 1 30); do
  FRONTEND_URL=$(grep -Eo 'https://[-a-z0-9]+\.trycloudflare\.com' /content/local-mirofish/logs/tunnel-frontend.log | head -1 || true)
  [ -n "${FRONTEND_URL}" ] && break
  sleep 2
done
test -n "${FRONTEND_URL:-}"
echo "Open MiroFish: $FRONTEND_URL"
echo "API tunnel:     $BACKEND_URL"
echo 'LLM and embedding APIs remain private on the Colab VM.'

## Diagnostics

Run this cell if graph building or simulation fails. Never publish full logs if they include seed content.

- A vLLM 400/500 often indicates that the small model could not honor a complex schema. Try a smaller input and simulation.
- A Neo4j error means inspect the Neo4j log and confirm its container is running.
- If `uv sync --extra graphiti` failed, do not continue: the backend cannot replace Zep until Graphiti is installed.
- A browser request to `localhost:5001` means the Vite API environment variable was not consumed by the revision you cloned; inspect the frontend log/source for its configured API variable.

In [ ]:
%%bash
set +e
echo '--- GPU ---'; nvidia-smi
echo '--- local LLM ---'; tail -n 80 /content/local-mirofish/logs/vllm.log
echo '--- embeddings ---'; tail -n 60 /content/local-mirofish/logs/embed.log
echo '--- MiroFish backend ---'; tail -n 160 /content/local-mirofish/logs/mirofish-backend.log
echo '--- Neo4j ---'; docker logs --tail 100 mirofish-neo4j
echo '--- process/ports ---'; ss -ltnp | grep -E ':(3000|5001|7474|7687|8000|8001|9000)' || true

## Cleanup

Stops every service and removes the Colab-local code/data directory. Run this when you finish or before handing the notebook to someone else.

In [ ]:
%%bash
set +e
pkill -f cloudflared
pkill -f 'vllm.entrypoints.openai.api_server'
pkill -f uvicorn
pkill -f 'flask run'
pkill -f vite
docker rm -f mirofish-neo4j
docker volume rm mirofish_neo4j_data
rm -rf /content/MiroFish-local /content/local-mirofish
echo 'Stopped services and removed VM-local MiroFish, Neo4j data, model-server logs, and tunnel state.'